# Data Understanding and Validation

This notebook inspects the cleaned infrastructure, NHM budget, and HMIS datasets. It performs descriptive checks only: no rows are modified or removed, no missing values or years are filled, and no machine-learning models are trained.

In [10]:
from pathlib import Path

import pandas as pd

current_working_directory = Path.cwd().resolve()
root_candidates = (current_working_directory, *current_working_directory.parents)
project_root = next(
    (path for path in root_candidates if (path / "data" / "cleaned").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not locate the VS Code project root from the current working "
        f"directory: {current_working_directory}"
    )
project_root = project_root.resolve()

file_paths = {
    "Infrastructure": project_root / "data" / "cleaned" / "infrastructure_state_year.csv",
    "Budget": project_root / "data" / "cleaned" / "nhm_budget_long_corrected.csv",
    "HMIS": project_root / "data" / "cleaned" / "hmis_long_normalized.csv",
}

print(f"Current working directory: {current_working_directory}")
print(f"Absolute project root: {project_root}")
print("CSV file diagnostics:")
for dataset_name, file_path in file_paths.items():
    exists = file_path.is_file()
    file_size = file_path.stat().st_size if exists else None
    size_text = f"{file_size} bytes" if exists else "N/A (file does not exist)"
    print(
        f"{dataset_name}: {file_path} | exists: {exists} | "
        f"size: {size_text}"
    )


Current working directory: C:\Users\bagad\health-budget-ml\notebooks
 Absolute project root: C:\Users\bagad\health-budget-ml
 CSV file diagnostics:
 Infrastructure: C:\Users\bagad\health-budget-ml\data\cleaned\infrastructure_state_year.csv | exists: True | size: 0 bytes
 Budget: C:\Users\bagad\health-budget-ml\data\cleaned\nhm_budget_long_corrected.csv | exists: True | size: 0 bytes
 HMIS: C:\Users\bagad\health-budget-ml\data\cleaned\hmis_long_normalized.csv | exists: True | size: 0 bytes


In [11]:
datasets = None
missing_files = [path for path in file_paths.values() if not path.is_file()]
if missing_files:
    missing_details = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(
        "Required cleaned CSV file(s) are missing:\n" + missing_details
    )

loaded_datasets = {}
load_errors = []
for dataset_name, file_path in file_paths.items():
    try:
        dataset = pd.read_csv(file_path)
    except pd.errors.EmptyDataError:
        load_errors.append(
            f"{dataset_name}: file exists but is empty and has no readable header or rows: {file_path}"
        )
        continue

    if dataset.shape[0] == 0 or dataset.shape[1] == 0:
        load_errors.append(
            f"{dataset_name}: pandas loaded shape {dataset.shape}, but actual rows and columns are required: {file_path}"
        )
        continue
    loaded_datasets[dataset_name] = dataset

if load_errors:
    raise ValueError(
        "Cannot continue until every cleaned CSV loads with actual rows and columns:\n"
        + "\n".join(f"- {message}" for message in load_errors)
    )

datasets = loaded_datasets
for dataset_name, dataset in datasets.items():
    print(f"{dataset_name}: loaded shape {dataset.shape}")
    print(f"Columns: {list(dataset.columns)}")


ValueError: Cannot continue until every cleaned CSV loads with actual rows and columns:
- Infrastructure: file exists but is empty and has no readable header or rows: C:\Users\bagad\health-budget-ml\data\cleaned\infrastructure_state_year.csv
- Budget: file exists but is empty and has no readable header or rows: C:\Users\bagad\health-budget-ml\data\cleaned\nhm_budget_long_corrected.csv
- HMIS: file exists but is empty and has no readable header or rows: C:\Users\bagad\health-budget-ml\data\cleaned\hmis_long_normalized.csv

In [ ]:
import itertools
import re

from IPython.display import display

quality_findings = {}


def columns_matching(df, pattern):
    return [column for column in df.columns if re.search(pattern, str(column).lower())]


def inspect_dataset(dataset_name, df):
    print(f"## {dataset_name}")
    print(f"Shape: {df.shape}")
    print("First 5 rows:")
    display(df.head(5))
    print(f"Column names: {list(df.columns)}")
    print("Data types:")
    display(df.dtypes.rename("dtype").to_frame())

    missing = df.isna().sum().to_frame("missing_count")
    missing["missing_percent"] = (
        df.isna().mean().mul(100) if len(df) else float("nan")
    )
    print("Missing values and percentages:")
    display(missing)

    exact_duplicates = int(df.duplicated().sum())
    print(f"Duplicate rows (all columns): {exact_duplicates}")

    state_columns = columns_matching(df, r"state")
    if state_columns and len(df):
        for column in state_columns:
            states = df[column].dropna().drop_duplicates().tolist()
            print(f"Unique values in {column!r} ({len(states)}): {states}")
    else:
        print("Unique states: unavailable (no state column or no data rows).")

    period_columns = columns_matching(
        df, r"year|period|month|quarter|financial|fiscal|date"
    )
    if period_columns and len(df):
        for column in period_columns:
            periods = df[column].dropna().drop_duplicates().tolist()
            print(f"Available values in {column!r} ({len(periods)}): {periods}")
    else:
        print("Available years/reporting periods: unavailable (no matching column or no data rows).")

    numeric_columns = df.select_dtypes(include="number").columns
    print("Descriptive statistics for numerical columns:")
    if len(numeric_columns):
        display(df[numeric_columns].describe().T)
    else:
        print("No numerical columns available.")

    quality_findings[dataset_name] = {
        "shape": df.shape,
        "empty_file": df.shape == (0, 0),
        "exact_duplicate_rows": exact_duplicates,
        "missing_percent": missing["missing_percent"].to_dict(),
    }


def check_state_year_duplicates(dataset_name, df):
    state_columns = columns_matching(df, r"state")
    year_columns = columns_matching(df, r"year|financial|fiscal")
    if not state_columns or not year_columns:
        print(f"{dataset_name} State-Year duplicate check: unavailable; state/year columns not identified.")
        quality_findings[dataset_name]["state_year_duplicate_rows"] = None
        return
    if df.empty:
        print(f"{dataset_name} State-Year duplicate check: unavailable; no data rows.")
        quality_findings[dataset_name]["state_year_duplicate_rows"] = None
        return

    state_column = state_columns[0]
    year_column = year_columns[0]
    key = [state_column, year_column]
    duplicate_mask = df.duplicated(subset=key, keep=False)
    duplicate_rows = int(duplicate_mask.sum())
    duplicate_keys = int(df.loc[duplicate_mask, key].drop_duplicates().shape[0])
    print(
        f"{dataset_name} duplicate {key} combinations: "
        f"{duplicate_keys} keys across {duplicate_rows} rows."
    )
    if duplicate_rows:
        display(df.loc[duplicate_mask, key].drop_duplicates().head(20))
    quality_findings[dataset_name]["state_year_duplicate_rows"] = duplicate_rows


def identify_hmis_observation_key(df):
    if df.empty:
        print("HMIS observation key: cannot be identified because there are no data rows.")
        quality_findings["HMIS"]["observation_key"] = None
        quality_findings["HMIS"]["observation_key_duplicate_rows"] = None
        return

    dimension_pattern = (
        r"state|district|block|subdistrict|facility|hospital|institution|"
        r"indicator|metric|service|program|scheme|year|period|month|quarter|"
        r"financial|fiscal|date|gender|sex|age|category|group|level|unit"
    )
    measure_pattern = r"value|count|total|rate|percent|numerator|denominator|amount"
    candidates = [
        column
        for column in df.columns
        if re.search(dimension_pattern, str(column).lower())
        and not re.search(measure_pattern, str(column).lower())
    ]
    if not candidates:
        candidates = [
            column
            for column in df.columns
            if not re.search(measure_pattern, str(column).lower())
        ]

    minimal_keys = []
    maximum_key_width = min(5, len(candidates))
    for width in range(1, maximum_key_width + 1):
        for key_tuple in itertools.combinations(candidates, width):
            if any(set(existing).issubset(key_tuple) for existing in minimal_keys):
                continue
            key = list(key_tuple)
            complete_rows = df[key].notna().all(axis=1)
            complete_data = df.loc[complete_rows, key]
            if not complete_data.empty and not complete_data.duplicated().any():
                minimal_keys.append(key_tuple)

    if not minimal_keys:
        print(
            "HMIS observation key: no unique composite key found among the "
            f"candidate columns (tested up to {maximum_key_width} columns)."
        )
        duplicate_rows = int(df.duplicated().sum())
        print(f"Exact duplicate HMIS rows across all columns: {duplicate_rows}")
        quality_findings["HMIS"]["observation_key"] = None
        quality_findings["HMIS"]["observation_key_duplicate_rows"] = duplicate_rows
        return

    print("HMIS minimal candidate column(s) that uniquely identify complete observations:")
    for key_tuple in minimal_keys:
        key = list(key_tuple)
        complete_rows = df[key].notna().all(axis=1)
        incomplete_key_rows = int((~complete_rows).sum())
        duplicate_rows = int(df.loc[complete_rows, key].duplicated(keep=False).sum())
        print(
            f"{key}: {int(complete_rows.sum())} complete rows; "
            f"{incomplete_key_rows} rows have missing key values; "
            f"{duplicate_rows} duplicate rows among complete keys."
        )
    quality_findings["HMIS"]["observation_key"] = [list(key) for key in minimal_keys]
    quality_findings["HMIS"]["observation_key_duplicate_rows"] = 0

## Infrastructure Dataset

Review the infrastructure dataset's dimensions, schema, missingness, duplicate rows, state and reporting-period values, and numerical summaries. Then check whether each State-Year combination is unique.


In [ ]:
inspect_dataset("Infrastructure", datasets["Infrastructure"])
check_state_year_duplicates("Infrastructure", datasets["Infrastructure"])

## NHM Budget Dataset

Review the budget dataset's dimensions, schema, missingness, duplicate rows, state and reporting-period values, and numerical summaries. Then check whether each State-Year combination is unique.


In [ ]:
inspect_dataset("Budget", datasets["Budget"])
check_state_year_duplicates("Budget", datasets["Budget"])

## HMIS Dataset

Review the HMIS dataset's dimensions, schema, missingness, duplicate rows, state and reporting-period values, and numerical summaries. Candidate observation keys are inferred from identifier and reporting-dimension column names, then tested for uniqueness among rows with complete key values.


In [ ]:
inspect_dataset("HMIS", datasets["HMIS"])
identify_hmis_observation_key(datasets["HMIS"])

## Data Quality Summary

Print a concise status for each source file and the dataset-specific key checks. Empty files are reported as unavailable for data-level validation rather than as evidence of zero missingness or duplicates.

In [ ]:
print("Data Quality Summary")
for dataset_name, findings in quality_findings.items():
    row_count, column_count = findings["shape"]
    dataset = datasets[dataset_name]
    if dataset.empty:
        source_state = "headerless empty file" if findings["empty_file"] else "no data rows"
        print(
            f"- {dataset_name}: {source_state} (shape {row_count} x {column_count}); "
            "data-level missingness, duplicate, state, period, and key checks are unavailable."
        )
        continue

    total_missing = int(dataset.isna().sum().sum())
    total_cells = row_count * column_count
    missing_percent = total_missing / total_cells * 100 if total_cells else float("nan")
    print(
        f"- {dataset_name}: shape {row_count} x {column_count}; "
        f"{total_missing} missing values ({missing_percent:.2f}% of cells); "
        f"{findings['exact_duplicate_rows']} exact duplicate rows."
    )
    if dataset_name in ("Infrastructure", "Budget"):
        print(
            f"  State-Year duplicate rows: "
            f"{findings.get('state_year_duplicate_rows', 'not checked')}"
        )
    if dataset_name == "HMIS":
        print(
            f"  Candidate observation key(s): "
            f"{findings.get('observation_key') or 'not identified'}; "
            f"duplicate rows for key check: "
            f"{findings.get('observation_key_duplicate_rows', 'not checked')}"
        )